In [63]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv

In [ ]:
from pathlib import Path
def extract_features(file, label): #定義特徵函數(檔案名稱，分類)
    data = pd.read_csv(file, encoding = "utf-8")

    features = {}

#基線校正

    #先切出baseline區間
    data["datetime"] = pd.to_datetime(data["datetime"])
    data["time_sec"] = (data["datetime"]-data["datetime"].iloc[0]).dt.total_seconds() #先將資料的時間修到剩秒數
    baseline = data[(data["time_sec"]>=20) & (data["time_sec"]<30)] #取baseline為進氣前10秒

    for i in range(1,17):
        channel = f"ch{i}" #用f即可自動將變數放到字串裡
        corrected_channel = f"{channel}_corrected" #令修正完時間後的channel為corrected_channel
        baseline_mean = baseline[channel].mean() #將baseline取平均
        data[corrected_channel] = data[channel]-baseline_mean #全曲線-baseline


    #切區間
    response = data[data["step"] == "Measure"]
    recovery = data[data["step"] == "Clean"]

#特徵設計
    #delta_f: response 區間的穩態頻率變化量
    #設穩態值為response前後各切10秒的平均，這裡即為40~140秒

    for i in range(1,17):
        channel = f"ch{i}"
        corrected_channel = f"{channel}_corrected"
        steady = response[(response["time_sec"]>=40)&(response["time_sec"]<140)] #在40~140秒內量測到的頻率設為穩態區間
        delta_f = steady[corrected_channel].mean() #將穩態區間的平均變化量取平均()
        features[f"{channel}_delta_f"] = float(delta_f) #轉成浮點數

    #response_slope: 反應初期的斜率
    #在response初期，QCM校正後頻率變化率；在晶體吸附VOC後，共振頻率下降，斜率為負數
    response_initial = response[(response["time_sec"]>=40)&(response["time_sec"]<50)] #取response的前10秒數據
    for i in range(1,17):
        channel = f"ch{i}"
        corrected = f"{channel}_corrected"

        x_response = response_initial["time_sec"] 
        y_response = response_initial[corrected]
        response_slope, _ = np.polyfit(x_response, y_response, 1)  #算出response的斜率和截距

        features[f"{channel}_response_slope"] = float(response_slope) 
    

    #recovery_slope: 恢復期的斜率
    #在recovery初期，頻率變化率；原本吸附的VOC離開感測層，即為感測層的恢復速度，斜率為正數
    recovery_initial = recovery[(recovery["time_sec"]>=150)&(recovery["time_sec"]<160)] #取recovery的前10秒數據
    for i in range(1,17):
        channel = f"ch{i}"
        corrected = f"{channel}_corrected"

        x_recovery = recovery_initial["time_sec"]
        y_recovery = recovery_initial[corrected]
        recovery_slope, _ = np.polyfit(x_recovery, y_recovery, 1) #算出recovery的斜率和截距
        features[f"{channel}_recovery_slope"] = float(recovery_slope)


    #AUC: response區間的曲線下面積
    #有效的反應區間累積的反應量
    for i in range(1,17):
        channel = f"ch{i}"
        corrected = f"{channel}_corrected"
        auc = np.trapezoid(steady[corrected], steady["time_sec"]) #利用trapezoid函數算出曲線下面積
        features[f"{channel}_auc"] = float(auc)

    #算t90: 達到90%穩態值所需時間
    #這顆sensor要花多久，才能"幾乎完成"他的response?
    for i in range(1,17):
        channel = f"ch{i}"
        corrected = f"{channel}_corrected"
        steady_value = steady[corrected].mean() #將在穩態區間的頻率取平均
        threshold_90 = steady_value*0.9 #標準為90%

        #判斷steady_value正負(因為response有可能是正負)
        #reach_90: 所有有達到90%的資料
        if steady_value <0: 
            reach_90 = response[response[corrected] <= threshold_90] 
        else: 
            reach_90 = response[response[corrected] >= threshold_90]

        response_start = response["time_sec"].iloc[0] #取response開始的第一筆資料
        
        if len(reach_90)>0: #如果剛剛取到的資料數量有大於0 (看有沒有成功找到資料)
            t90_time = reach_90["time_sec"].iloc[0] #有的話就取第一筆資料!
            t90 = t90_time-response_start #拿到的秒數減掉response開始的時間，就可以知道是第幾秒達到90%
        else:
            t90 = np.nan#沒有取道的話就回傳NaN
            
        features[f"{channel}_t90"] = float(t90)

    path = Path(file) #從檔案路徑自動取得sample資訊
    sample_id = path.stem #.stem取得檔案名稱，不要副檔名
    subject_id = sample_id.split("-")[0] #遇到"-"就切開來，形成一個list，取裡面的第一個

    features["subject_id"] = subject_id #把剛剛得到的都放到字典裡
    features["sample_id"] = sample_id
    features["label"] = label

    return features #回傳給features這個字典

In [ ]:
files = ["005/005-1.csv", "007/007-1.csv", "008/008-1.csv"]

all_features = [] #建立一個空的list, 等等收集每個sample的分析結果

for file in files:
    features = extract_features(file, "CRC") #對目前的檔案進行特徵萃取，將label標為CRC
    all_features.append(features) #把得到的每個features一一加進去

In [ ]:
#把所有sample的features轉成DataFrame
feature_matrix = pd.DataFrame(all_features) #pd.DataFrame會整理成表格

#把sample資訊移到最前面
info_columns = ["subject_id", "sample_id", "label"] #建立一個list

#feature_matrix.columns代表feature_matrix裡面的所有欄位名稱
feature_columns = [
    col for col in feature_matrix.columns  #一個一個把欄位名稱拿出來，叫col
    if col not in info_columns #只有上面那些不會被存進feature_columns
]

#真正重新排列feature_matrix的欄位
feature_matrix = feature_matrix[info_columns + feature_columns]
#輸出csv
feature_matrix.to_csv("features.csv", index=False)